# 01 · Vulnerability scan → remediation tracker → dashboard

**Problem.** An insurer's security team gets a quarterly vulnerability scan (plus ad-hoc rescans after patching) as a large spreadsheet export. Each quarter, analysts filtered it by hand, matched servers to the CMDB, copied findings into per-entity Excel trackers, and kept owner notes and exceptions from the last quarter.

**What was automated.** The production solution runs on Microsoft Power Platform (SharePoint, Power Automate, Office Scripts, Dataverse, Power BI) with GitHub Copilot / Copilot Studio agents for questions in plain language. This notebook reproduces the **same logic in Python** so it can be shown publicly.

> All data here is **synthetic** (a fictitious "Example Insurance Group", `corp.example.com`). It was generated to reproduce the data problems found in real scanner exports, not copied from them.

**Steps**
1. Load the scans and the CMDB
2. Normalise asset names and match them to the CMDB (hostname, then IP fallback)
3. Remove stale duplicate records (the scanner re-keys assets)
4. Keep actionable severities (Medium and above)
5. Build the Q2 tracker, apply the ad-hoc rescan safely, carry forward to Q3
6. Dashboard charts (the Power BI equivalent)
7. Export per-entity trackers, and a small "ask the agent" query function

In [ ]:
import os
REPO = "https://github.com/tantg88888/secops-automation-demo"
if not os.path.exists("data"):                      # running in Colab: fetch the repo
    if not os.path.exists("secops-automation-demo"):
        os.system(f"git clone -q {REPO}")
    os.chdir("secops-automation-demo")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)
os.makedirs("images", exist_ok=True)
os.makedirs("output", exist_ok=True)

## 1 · Load

In [ ]:
cmdb  = pd.read_csv("data/cmdb_assets.csv")
q2    = pd.read_csv("data/scan_2026Q2.csv")
q3    = pd.read_csv("data/scan_2026Q3.csv")
adhoc = pd.read_csv("data/scan_adhoc_2026-08-14.csv")
print(f"CMDB {len(cmdb)} assets | Q2 {len(q2)} rows | Q3 {len(q3)} rows | ad-hoc {len(adhoc)} rows")
q2.head()

## 2 · Normalise asset names and match to the CMDB
The same server appears as an FQDN, an upper-case short name, or just an IP address. Matching on the raw name misses many assets, so names are normalised first and IP is used as a fallback.

In [ ]:
def norm_host(name):
    s = str(name).strip().lower()
    return s if s.replace(".", "").isdigit() else s.split(".")[0]

cmdb["host_key"] = cmdb["hostname"].map(norm_host)
asset_by_host = cmdb.set_index("host_key")["asset_id"]
asset_by_ip   = cmdb.set_index("ip_address")["asset_id"]
CMDB_COLS = ["asset_id", "entity", "criticality", "environment", "owner_team"]

def match_cmdb(df):
    df = df.copy()
    df["host_key"] = df["asset_name"].map(norm_host)
    by_host = df["host_key"].map(asset_by_host)
    by_ip   = df["ipv4"].map(asset_by_ip)
    df["match_method"] = np.select([by_host.notna(), by_ip.notna()], ["hostname", "ip"], "unmatched")
    df["asset_id"] = by_host.fillna(by_ip)
    return df.merge(cmdb[CMDB_COLS], on="asset_id", how="left")

naive = q2["asset_name"].isin(cmdb["hostname"]).mean()
q2m, q3m, adm = match_cmdb(q2), match_cmdb(q3), match_cmdb(adhoc)
print(f"Raw-name match rate: {naive:.0%}")
print(q2m["match_method"].value_counts().to_string())
print("\nUnknown to the CMDB (raise with asset owners):", sorted(q2m.loc[q2m.match_method == "unmatched", "asset_name"].unique()))

## 3 · Remove stale duplicates
The scanner sometimes re-keys an asset, so one real finding shows up twice: an old record and a new one with a different `finding_id`. Counting by `finding_id` double-counts. A finding is identified by **asset | plugin | port**, keeping the most recent record.

In [ ]:
ACTIONABLE = ["Critical", "High", "Medium"]

def dedupe(df, label):
    df = df.copy()
    df["asset_key"]   = df["asset_id"].fillna(df["host_key"])
    df["finding_key"] = df["asset_key"] + "|" + df["plugin_id"].astype(str) + "|" + df["port"].astype(str)
    df["last_seen"]   = pd.to_datetime(df["last_seen"])
    out = df.sort_values("last_seen").drop_duplicates("finding_key", keep="last")
    before = df["severity"].isin(ACTIONABLE).sum()
    after  = out["severity"].isin(ACTIONABLE).sum()
    print(f"{label}: {len(df) - len(out)} stale duplicates removed; Medium+ {before} -> {after}")
    return out

q2d, q3d, add = dedupe(q2m, "Q2"), dedupe(q3m, "Q3"), dedupe(adm, "Ad-hoc")

## 4 · Severity: reported vs vendor
Findings with no available fix are re-cast to *Info* by the scanner policy. The tracker carries only actionable items (Medium and above); the vendor severity is kept for reference.

In [ ]:
display(pd.crosstab(q2d["vendor_severity"], q2d["severity"]))
q2a, q3a = q2d[q2d.severity.isin(ACTIONABLE)], q3d[q3d.severity.isin(ACTIONABLE)]

## 5 · Tracker: Q2 → ad-hoc rescan → Q3
Rules agreed with the business:
* **IDs are not stable** between exports, so findings are carried forward on `finding_key`, never on `finding_id`.
* **Absence is not proof of a fix.** A finding missing from a later scan is flagged for verification; only a human closes it.
* An ad-hoc rescan with **local checks only (port 0)** cannot confirm network-port findings, so those stay open.

In [ ]:
same_id = len(set(q2a.finding_id) & set(q3a.finding_id))
same_key = len(set(q2a.finding_key) & set(q3a.finding_key))
print(f"Q2 findings still present in Q3: {same_key} by finding_key, {same_id} by finding_id")

tracker = q2a[["finding_key", "asset_key", "entity", "plugin_id", "plugin_name", "severity", "port", "owner_team"]].copy()
tracker["status"] = "Open"

# Ad-hoc rescan (Aug): only assets and checks it actually covered
rescanned = set(add["asset_key"])
local_only = add["port"].eq(0).all()
in_scope = tracker["asset_key"].isin(rescanned) & (tracker["port"].eq(0) if local_only else True)
still_open = tracker["finding_key"].isin(add["finding_key"])
tracker.loc[in_scope & ~still_open, "status"] = "Fix indicated by ad-hoc scan - verify"
tracker.loc[tracker["asset_key"].isin(rescanned) & ~in_scope, "status"] = "Open (not covered by ad-hoc scan)"

# Q3 carry-forward
q3_keys = set(q3a["finding_key"])
tracker.loc[~tracker["finding_key"].isin(q3_keys) & tracker["status"].eq("Open"), "status"] = "Not seen in Q3 - verify"
new_q3 = q3a[~q3a["finding_key"].isin(tracker["finding_key"])].assign(status="New in Q3")
tracker = pd.concat([tracker, new_q3[tracker.columns]], ignore_index=True)
tracker["status"].value_counts()

## 6 · Dashboard (Power BI equivalent)

In [ ]:
SEV_COLORS = {"Critical": "#8B1E3F", "High": "#D1495B", "Medium": "#EDAE49"}
open_now = tracker[~tracker["status"].str.startswith(("Not seen", "Fix indicated"))]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
(open_now.pivot_table(index="entity", columns="severity", values="finding_key", aggfunc="count")
         .reindex(columns=ACTIONABLE).fillna(0)
         .plot.barh(stacked=True, ax=axes[0], color=[SEV_COLORS[s] for s in ACTIONABLE]))
axes[0].set_title("Open Medium+ findings by entity"); axes[0].set_xlabel("findings"); axes[0].set_ylabel("")

trend = pd.DataFrame({"FY26 Q2": q2a.severity.value_counts(), "FY26 Q3": q3a.severity.value_counts()}).reindex(ACTIONABLE)
trend.T.plot.bar(ax=axes[1], color=[SEV_COLORS[s] for s in ACTIONABLE], rot=0)
axes[1].set_title("Quarter-on-quarter trend"); axes[1].set_ylabel("findings")

top = open_now.groupby("asset_key").size().nlargest(10).sort_values()
top.plot.barh(ax=axes[2], color="#30638E")
axes[2].set_title("Top 10 assets by open findings"); axes[2].set_ylabel("")

for ax in axes: ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig("images/vuln_dashboard.png", dpi=150); plt.show()

## 7 · Outputs: per-entity trackers, and a query tool for the agent
In production the agent calls a workflow with a small JSON query; this function plays that role.

In [ ]:
with pd.ExcelWriter("output/remediation_tracker_FY2026Q3.xlsx") as xw:
    for ent, grp in tracker.groupby("entity", dropna=False):
        grp.to_excel(xw, sheet_name=str(ent if pd.notna(ent) else "UNMATCHED")[:31], index=False)
print("Saved output/remediation_tracker_FY2026Q3.xlsx")

def open_findings(entity=None, min_severity="High", top=5):
    """e.g. 'Which High+ findings are open for EIG-LIFE?'"""
    rank = {s: i for i, s in enumerate(ACTIONABLE)}
    df = open_now[open_now["severity"].map(rank) <= rank[min_severity]]
    if entity:
        df = df[df["entity"] == entity]
    summary = df.groupby(["asset_key", "severity"]).size().unstack(fill_value=0)
    return f"{len(df)} open {min_severity}+ findings" + (f" for {entity}" if entity else ""), summary.head(top)

answer, table = open_findings("EIG-LIFE", "High")
print(answer); table